# Flipping a biased coin

A biased coin lands on **heads** with probability $p$ and **tails** with probability $1-p$. Each flip is _independent_. We encode heads as $1$ and tails as $0$.

For $n$ flips, the expected number of heads is $np$, but an individual experiment usually differs from this expectation. The histogram below counts the two possible outcomes in one sequence of $n$ flips.

**Run all cells**, then adjust **P(heads)** and **Number of flips** and click **Flip coins**. Click again for a fresh experiment. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# Store the most recent experiment for exploration in later cells.
flips = np.array([], dtype=np.int8)
experiment_p = None

## Simulation and summary functions

`simulate_flips(p, n, seed)` returns a NumPy array of zeros (tails) and ones (heads). 

In [2]:
def simulate_flips(p, n, seed=None):
    """Simulate n independent flips; 1 = heads, 0 = tails."""
    if not np.isfinite(p) or not 0 <= p <= 1:
        raise ValueError("p must be between 0 and 1.")
    if isinstance(n, (bool, np.bool_)) or not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError("n must be a positive integer.")
    return np.random.default_rng(seed).binomial(1, p, size=n).astype(np.int8)


def summarize_flips(sequence):
    """Summarize a nonempty, one-dimensional sequence of zeros and ones."""
    values = np.asarray(sequence)
    if values.ndim != 1 or values.size == 0 or not np.isin(values, [0, 1]).all():
        raise ValueError("Provide a nonempty one-dimensional sequence of 0s and 1s.")
    heads = int(values.sum())
    return {
        "n": int(values.size),
        "heads": heads,
        "tails": int(values.size - heads),
        "proportion_heads": float(values.mean()),
        "proportion_tails": float(1 - values.mean()),
        "empirical_variance": float(values.var(ddof=0)),
        "empirical_std": float(values.std(ddof=0)),
    }


In [4]:
p_control = widgets.FloatSlider(
    value=0.7, min=0, max=1, step=0.01, description="P(heads):",
    readout_format=".2f", continuous_update=False,
)
n_control = widgets.IntSlider(
    value=100, min=1, max=10000, step=1, description="Number of flips:",
    continuous_update=False, style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

show_stats = widgets.Checkbox(value=True, description="Show summary statistics")
show_sequence = widgets.Checkbox(value=True, description="Show flip sequence")
flip_button = widgets.Button(description="Flip coins", button_style="primary", icon="refresh")
output = widgets.Output()


def render_experiment():
    stats = summarize_flips(flips)
    n = stats["n"]
    p = experiment_p
    with output:
        output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(flips, bins=[-0.5, 0.5, 1.5], rwidth=0.65,
                color="#3575AB", edgecolor="white", label="Observed count")
        for x, count in enumerate([stats["tails"], stats["heads"]]):
            ax.annotate(str(count), (x, count), xytext=(0, 6),
                        textcoords="offset points", ha="center")
        ax.set(xticks=[0, 1], xticklabels=["Tails (0)", "Heads (1)"],
               ylabel="Number of flips", xlabel="Outcome",
               title=f"{n:,} independent flips · P(heads) = {p:.2f}")
        ax.set_ylim(0, max(stats["heads"], stats["tails"], n * p, n * (1 - p)) * 1.25 + 1)
        ax.yaxis.get_major_locator().set_params(integer=True)
        fig.tight_layout()
        plt.show()
        plt.close(fig)

        if show_sequence.value:
            preview = " ".join("H" if value else "T" for value in flips[:100])
            label = f"First 100 of {n:,} flips" if n > 100 else f"All {n} flips"
            display(Markdown(f"**{label}** (H = heads, T = tails)\n\n`{preview}`"))
            display(Markdown("The full sequence is available in the variable `flips`."))


def run_experiment(_=None):
    global flips, experiment_p
    experiment_p = p_control.value
    seed = None
    flips = simulate_flips(experiment_p, n_control.value, seed)
    render_experiment()

def update_seed_control(change):
    seed_control.disabled = not change["new"]

flip_button.on_click(run_experiment)
show_stats.observe(lambda change: render_experiment(), names="value")
show_sequence.observe(lambda change: render_experiment(), names="value")
display(widgets.VBox([
    p_control, n_control,
    widgets.HBox([show_stats, show_sequence]),
    flip_button,
    widgets.HTML("Changes to p, n, or the seed take effect when you click <b>Flip coins</b>."),
    output,
]))
run_experiment()
